# TeleRAG-Agent: QLoRA Fine-Tuning on Kaggle (T4)
This notebook uses **Unsloth** for 2-5x faster training and 80% less memory usage.
**Fixed for AttributeError: `int` object has no attribute `mean`**

In [ ]:
# Install dependencies (Unsloth, TRL, etc)
!pip install unsloth transformers datasets trl accelerate bitsandbytes xformers -q

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
from kaggle_secrets import UserSecretsClient

# ----- Configuration -----
max_seq_length = 2048
dtype = None
load_in_4bit = True

# ----- Load HF Token -----
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
except:
    hf_token = "" # Fallback if not running in Kaggle environment with secrets

In [ ]:
# ----- Load Model -----
model_name = "AliMaatouk/LLama-3-8B-Tele-it"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    token=hf_token,
)

# ----- CRITICAL: Set pad_token (fixes the mean error) -----
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# ----- Add LoRA adapters -----
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
# ----- Load Dataset (from your Kaggle Dataset) -----
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/teleqna-processed/teleqna_train.jsonl",
    split="train"
)

# ----- FIXED prompt template (includes INPUT field!) -----
# CRITICAL FIX: Previous version only used instruction + output,
# completely ignoring the 'input' field which contains the actual
# question and options. The model learned output FORMAT but not REASONING.
#
# Correct Llama-3 chat template:
#   system: instruction (domain expert prompt)
#   user: input (question + options)
#   assistant: output (answer)

def formatting_prompts_func(examples):
    texts = []
    for instr, inp, out in zip(examples['instruction'], examples['input'], examples['output']):
        if not instr or not inp or not out:
            continue
        text = (
            f"<|start_header_id|>system<|end_header_id|>\n\n"
            f"{instr}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{inp}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{out}<|eot_id|>"
        )
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

# Verify a sample
print("Sample formatted text:")
print(dataset[0]['text'][:500])
print(f"\nTotal training samples: {len(dataset)}")


### 100-Sample Test Run (Run this first!)
Verifies that everything works without OOM before doing the full multi-hour training loop.

In [ ]:
# ----- Test Run (100 samples, 20 steps) -----
test_dataset = dataset.select(range(100))

trainer_test = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=test_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=20,
        learning_rate=2e-4,
        fp16=torch.cuda.is_available(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        output_dir="outputs_test",
        seed=3407,
        # Critical: ensures labels are set correctly
        remove_unused_columns=False,
    ),
)

print("🚀 Running test training (20 steps)...")
trainer_test.train()
print("✅ Test training completed!")

### Full Training Run
Only run this once the test run succeeds. Takes ~2-3 hours on T4.

In [ ]:
# ----- Full Training (3 epochs) -----
trainer_full = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        optim="adamw_8bit",
        weight_decay=0.01,
        output_dir="outputs",
        seed=3407,
        remove_unused_columns=False,
    ),
)

print("🚀 Starting full training (3 epochs)...")
trainer_full.train()
print("✅ Full training completed!")

In [ ]:
# ----- Save and zip adapter -----
model.save_pretrained("telerag_lora_model")
tokenizer.save_pretrained("telerag_lora_model")
!zip -r telerag_lora_model.zip telerag_lora_model/
print("📦 Adapter saved as `telerag_lora_model.zip` - download from Kaggle output.")

# (Optional) Push to Hugging Face
# model.push_to_hub("chaitanya-k/TeleRAG-LoRA", token=hf_token)
# tokenizer.push_to_hub("chaitanya-k/TeleRAG-LoRA", token=hf_token)